 # Mechanism algorithms tutorial



 ## Running the provided fishery benchmark with quota, incentives, social influence, and composition



 This is a **VS Code / Jupyter percent-format notebook**.



 Save as:



 `mechanism_benchmarks_tutorial.py`



 VS Code recognizes `# %%` as notebook cells. Use:



 **Jupyter: Export Current Python File as Jupyter Notebook**



 to generate an `.ipynb`.



 ---



 ## Important branch status



 This tutorial documents the **intended public abstraction** of the mechanism

 refactor. The architecture is implemented, but this branch still requires

 end-to-end integration fixes and validation against `dev`.



 Before treating the code cells below as production-ready, reconcile:



 - `BilevelConfig.mechanism(...)` with the new mechanism-object API;

 - missing abstract-method implementations on concrete mechanisms;

 - reward/observation dispatch in `MultiAgentRegulatedEnv`;

 - final action/observation space dimensions;

 - restoration dynamics and subsidy action semantics;

 - the currently partial social-influence implementation.



 The companion `MECHANISM_ABSTRACTION_TODO.md` lists the concrete fixes.

 # 1. Reinforcement learning before mechanisms



 A Markov decision process can be written as:



 \[

 \mathcal E

 =

 (\mathcal S,\mathcal A,P,R,\gamma).

 \]



 At time \(t\):



 1. the environment is in state \(s_t\);

 2. the agent receives observation \(o_t\);

 3. the policy samples:



 \[

 a_t \sim \pi_\phi(\cdot\mid o_t);

 \]



 4. the environment transitions:



 \[

 s_{t+1}\sim P(\cdot\mid s_t,a_t);

 \]



 5. the agent receives reward \(r_t\).



 A Bellman-style value equation is:



 \[

 V^{\pi_\phi}(s_t)

 =

 \mathbb E

 \left[

 r_t+\gamma V^{\pi_\phi}(s_{t+1})

 \right].

 \]



 The learned policy depends on the experience generated by this loop.

 Therefore, changing observations, actions, or rewards changes the policy

 optimization problem.

 # 2. Mechanism intervention



 We represent a mechanism as:



 \[

 \mathcal M_\theta

 =

 \left(

 \mathcal M_\theta^O,

 \mathcal M_\theta^A,

 \mathcal M_\theta^R

 \right).

 \]



 Observation shaping:



 \[

 o_t^\*=\mathcal M_\theta^O(s_t,o_t).

 \]



 The policy acts on:



 \[

 a_t\sim \pi_\phi(\cdot\mid o_t^\*).

 \]



 Action shaping:



 \[

 a_t^\*=\mathcal M_\theta^A(s_t,a_t).

 \]



 The environment transition receives \(a_t^\*\):



 \[

 s_{t+1}\sim P(\cdot\mid s_t,a_t^\*).

 \]



 Reward shaping:



 \[

 r_t^\*

 =

 \mathcal M_\theta^R(

 r_t,

 s_t,

 a_t^\*,

 s_{t+1}

 ).

 \]



 The learner therefore optimizes:



 \[

 V^{\pi_\phi,\theta}(s_t)

 =

 \mathbb E

 \left[

 r_t^\*+\gamma V^{\pi_\phi,\theta}(s_{t+1})

 \right].

 \]



 This separation lets a benchmark describe ecology/physics while a mechanism

 describes regulation.

 ## 2.1 How this maps to code



 The mechanism base class exposes:



 ```python

 mechanism.action(action_dict, ...)

 mechanism.observation(observation_dict, ...)

 mechanism.reward(reward_dict, ...)

 ```



 A mechanism may require benchmark-specific runtime values. Those are injected

 through **bindings**.



 Example:



 ```python

 bindings={

     "resource_level": lambda env: (

         env.S_t["fish"] / env.K

     )

 }

 ```



 `Mechanism.resolve(env)` converts the binding into:



 ```python

 {

     "resource_level": 0.73,

 }

 ```



 This lets the same quota algorithm operate on fish biomass, water level, or

 another normalized resource without hard-coding the benchmark.

 # 3. Fishery benchmark mathematics



 Let:



 \[

 B_t=\text{fish biomass}.

 \]



 The Pella-Tomlinson growth model is:



 \[

 G(B_t)

 =

 \frac{r}{p}

 B_t

 \left[

 1-

 \left(\frac{B_t}{K}\right)^p

 \right],

 \]



 where:



 - \(r\): intrinsic growth rate;

 - \(K\): carrying capacity;

 - \(p\): shape parameter;

 - \(p=1\): Schaefer/logistic case.



 With stochastic growth \(\epsilon_t\), restoration \(I_t\), and realized

 harvest \(H_t\):



 \[

 B_{t+1}

 =

 \operatorname{clip}

 \left(

 B_t+G(B_t)+\epsilon_t+I_t-H_t

 \right).

 \]



 The current benchmark uses a two-component agent action:



 \[

 a_{i,t}

 =

 \begin{bmatrix}

 h_{i,t}\\

 e_{i,t}

 \end{bmatrix},

 \]



 with:



 - \(h_{i,t}\): harvest fraction;

 - \(e_{i,t}\): restoration effort.

In [ ]:
import numpy as np

EPS = 1e-8


 # 4. Quota mechanism



 The quota is an **action-shaping mechanism**.



 It modifies the harvest action before the benchmark transition.

 ## 4.1 Mathematical formulation



 Define normalized biomass:



 \[

 b_t=\frac{B_t}{K}\in[0,1].

 \]



 Let the quota threshold be:



 \[

 q\in[0,1].

 \]



 With transition width \(w_q\):



 \[

 L=\sigma\left(\frac{0-q}{w_q}\right),

 \]



 \[

 U=\sigma\left(\frac{1-q}{w_q}\right),

 \]



 \[

 C_t=\sigma\left(\frac{b_t-q}{w_q}\right).

 \]



 The allowed action fraction is:



 \[

 \alpha_t

 =

 \frac{C_t-L}{U-L}.

 \]



 Interpretation:



 - resource far below threshold -> \(\alpha_t\approx 0\);

 - resource far above threshold -> \(\alpha_t\approx 1\);

 - near threshold -> smooth transition.



 For requested harvest fraction \(h_{i,t}\):



 \[

 h_{i,t}^{*}

 =

 h_{i,t}

 -

 \operatorname{smooth}_{+}

 \left(

 h_{i,t}-\alpha_t;

 w_u

 \right).

 \]



 This approximates:



 \[

 h_{i,t}^{*}\approx \min(h_{i,t},\alpha_t).

 \]



 Therefore:



 \[

 a_{i,t}^{*}

 =

 \begin{bmatrix}

 h_{i,t}^{*}\\

 e_{i,t}

 \end{bmatrix}.

 \]

 ## 4.2 Quota configuration variables



 ```text

 fixed_quota

     q in the equations above.



 action_component

     Which component is regulated.

     Fishery: 0 = harvest.



 quota_transition_width

     w_q. Smoothness around the quota threshold.



 usage_transition_width

     w_u. Smoothness of the action cap.



 violation_transition_width

     Available for smooth violation semantics; the shown action transform does

     not currently use it directly.



 bindings["resource_level"]

     Runtime normalized resource state b_t.

 ```

 ## 4.3 Constructing a quota mechanism



 The intended new-object API is:

In [ ]:
# Uncomment after P0 integration fixes are merged.
#
# from core.mechanism.algorithms.quota import QuotaMechanism
#
# quota = QuotaMechanism(
#     fixed_quota=0.56224,
#     action_component=0,
#     quota_transition_width=0.03,
#     usage_transition_width=0.005,
#     violation_transition_width=0.03,
#     bindings={
#         "resource_level": lambda env: (
#             env.S_t["fish"] / max(env.K, EPS)
#         ),
#     },
# )


 The same algorithm can be reused in a water benchmark:



 ```python

 bindings={

     "resource_level": lambda env: (

         env.S_t["reservoir"] / env.reservoir_capacity

     )

 }

 ```

 # 5. Learned incentive / restoration subsidy



 The subsidy is a **reward-shaping mechanism**.



 The benchmark determines the ecological effect of restoration.

 The mechanism determines the incentive attached to restoration.

 ## 5.1 General learned incentive



 A learned incentive designer can be represented as:



 \[

 u_t=\mu_\eta(s_t,a_t),

 \]



 and:



 \[

 R_\eta^{(i)}(s_t,a_t)

 =

 R^{(i)}

 \left(

 s_t,

 a_t,

 \mu_\eta(s_t,a_t)

 \right).

 \]



 In this codebase, the subsidy mechanism is simpler: the outer optimizer may

 optimize a subsidy coefficient. It is not itself a neural meta-gradient

 incentive network.

 ## 5.2 Fish habitat restoration incentive



 Let:



 - \(r_{i,t}\): base reward;

 - \(e_{i,t}\): restoration effort;

 - \(c\): cost coefficient;

 - \(\sigma_\theta\): subsidy.



 Then:



 \[

 r_{i,t}^{*}

 =

 r_{i,t}

 -

 c e_{i,t}^{2}

 +

 \sigma_\theta e_{i,t}.

 \]



 The quadratic cost discourages simply saturating restoration at its maximum.

 ## 5.3 Subsidy configuration



 ```text

 subsidy

     sigma_theta: linear incentive per unit restoration effort.



 cost

     c: quadratic restoration cost.



 action_component

     Fishery: 1 = restoration effort.

 ```

In [ ]:
# from core.mechanism.algorithms.subsidy import SubsidyMechanism
#
# subsidy = SubsidyMechanism(
#     subsidy=0.10,
#     cost=0.25,
#     action_component=1,
# )


 ## 5.4 Ecology versus incentive



 Ecological restoration belongs in the transition:



 \[

 I_t

 =

 \rho

 \sum_i e_{i,t},

 \]



 where \(\rho\) is restoration effectiveness.



 Then:



 \[

 B_{t+1}

 =

 B_t+G(B_t)+\epsilon_t+I_t-H_t.

 \]



 The subsidy changes reward:



 \[

 r_{i,t}

 \mapsto

 r_{i,t}-ce_{i,t}^2+\sigma_\theta e_{i,t}.

 \]



 These are separate mechanisms in the scientific model.

 # 6. Social influence



 Social influence can affect what agents observe and, in the full formulation,

 can add an intrinsic reward for influencing peer behavior.

 ## 6.1 Full social-influence objective



 For agent \(i\):



 \[

 c_t^i

 =

 \sum_{j\ne i}

 D_{KL}

 \left[

 \pi_j(a_t^j\mid a_t^i,s_t^j)

 \|

 \pi_j(a_t^j\mid s_t^j)

 \right].

 \]



 A social-influence reward is:



 \[

 r_t^i=r_{i,t}+\beta c_t^i.

 \]



 \(\beta\) controls influence strength.

 ## 6.2 What the current supplied implementation does



 The shown `SocialInfluenceMechanism` currently implements observation

 augmentation:



 \[

 o_{i,t}^{*}

 =

 [

 o_{i,t},

 a_{1,t-1},

 \dots,

 a_{j,t-1},

 \dots

 ],

 \qquad j\ne i.

 \]



 Each agent observes the previous actions of the other agents.



 This is an observation-shaping mechanism:



 \[

 \mathcal M_\theta^O.

 \]



 **The supplied code does not yet implement the KL reward bonus.**



 The shown `influence_weight` is also not used by the observation method. This

 is an explicit implementation TODO.

 ## 6.3 Social-influence bindings



 ```text

 previous_actions

     agent_id -> previous action vector.



 agent_ids

     Stable ordered set/tuple of participating agents.



 influence_weight

     beta; relevant once the reward bonus is implemented.

 ```

In [ ]:
# Target construction after the class exposes `bindings`.
#
# from core.mechanism.algorithms.social_influence import SocialInfluenceMechanism
#
# social = SocialInfluenceMechanism(
#     influence_weight=0.10,
#     bindings={
#         "previous_actions": lambda env: env.previous_actions,
#         "agent_ids": lambda env: tuple(env.agents),
#     },
# )


 ## 6.4 Observation dimension



 With:



 \[

 N=\text{number of agents},

 \qquad

 d_a=\text{action dimension},

 \]



 peer-action augmentation adds:



 \[

 (N-1)d_a

 \]



 features.



 The declared Gymnasium observation space must include this expansion.

 # 7. Optional threshold penalty



 The branch also contains a generic smooth reward penalty below a resource

 threshold.



 Let:



 - \(b_t\): normalized resource level;

 - \(\tau\): threshold;

 - \(\lambda\): maximum penalty;

 - \(w\): transition width.



 Then:



 \[

 p_t

 =

 \frac{\lambda}

 {1+\exp((b_t-\tau)/w)},

 \]



 and:



 \[

 r_{i,t}^{*}=r_{i,t}-p_t.

 \]

In [ ]:
# from core.mechanism.algorithms.penalty import ThresholdPenaltyMechanism
#
# penalty = ThresholdPenaltyMechanism(
#     threshold=0.20,
#     penalty_amount=0.10,
#     transition_width=0.03,
#     bindings={
#         "resource_level": lambda env: (
#             env.S_t["fish"] / max(env.K, EPS)
#         ),
#     },
# )


 # 8. Stacking mechanisms



 Composition lets several regulatory algorithms behave as one mechanism.



 There are two semantics:



 1. chained/sequential;

 2. parallel + merge.

 ## 8.1 Chained composition



 For:



 \[

 (M_1,M_2,\dots,M_k),

 \]



 chained action composition is:



 \[

 M_{\text{chain}}^A(x)

 =

 M_k^A(

 \dots

 M_2^A(

 M_1^A(x)

 )

 \dots

 ).

 \]



 The same ordering is used independently for reward and observation.



 Therefore order matters when two children alter the same channel.

 ### Example



 ```python

 children=(

     quota,

     subsidy,

     social,

 )

 ```



 Action channel:



 ```text

 Quota -> Subsidy(identity) -> Social(identity)

 ```



 Reward channel:



 ```text

 Quota(identity) -> Subsidy -> Social(identity in current code)

 ```



 Observation channel:



 ```text

 Quota -> Subsidy(identity) -> Social

 ```



 The final observation can therefore include quota information and peer-action

 information.

 ## 8.2 Optimizer representation



 If child dimensions are:



 \[

 d_1,\dots,d_k,

 \]



 then:



 \[

 d_{\text{chain}}=\sum_i d_i.

 \]



 `encode()` concatenates child vectors and `decode()` slices them back into

 the children.

In [ ]:
# from core.mechanism.composition.chained_mechanism import ChainedMechanism
#
# mechanism = ChainedMechanism(
#     children=(
#         quota,
#         subsidy,
#         social,
#     )
# )


 ## 8.3 Parallel composition



 In parallel composition every child receives the same original input:



 \[

 x

 \rightarrow

 \{

 M_1(x),

 M_2(x),

 \dots,

 M_k(x)

 \}.

 \]



 A merge operator combines outputs:



 \[

 M_{\parallel}(x)

 =

 \Gamma

 \left(

 x,

 M_1(x),

 \dots,

 M_k(x)

 \right).

 \]



 Separate merge operators may be used for action, reward, and observation.

In [ ]:
def additive_reward_merge(original, outputs):
    """Example: add each child's reward delta to the same base reward."""
    merged = {}

    for agent_id, base_reward in original.items():
        delta = sum(
            float(output[agent_id]) - float(base_reward)
            for output in outputs
        )
        merged[agent_id] = float(base_reward) + delta

    return merged


 Parallel composition is only order-independent if the merge function itself

 is order-independent.



 **Current branch note:** the supplied `ParallelMechanism` calls

 `child.apply_action/apply_reward/apply_observation`, while the supplied base

 class exposes `action/reward/observation`. Reconcile that API before using

 parallel composition.

 # 9. Building the fishery mechanism stack



 The intended construction is:

In [ ]:
# quota = QuotaMechanism(
#     fixed_quota=0.56224,
#     action_component=0,
#     bindings={
#         "resource_level": lambda env: (
#             env.S_t["fish"] / max(env.K, EPS)
#         ),
#     },
# )
#
# subsidy = SubsidyMechanism(
#     subsidy=0.10,
#     cost=0.25,
#     action_component=1,
# )
#
# social = SocialInfluenceMechanism(
#     influence_weight=0.10,
#     bindings={
#         "previous_actions": lambda env: env.previous_actions,
#         "agent_ids": lambda env: tuple(env.agents),
#     },
# )
#
# mechanism = ChainedMechanism(
#     children=(
#         quota,
#         subsidy,
#         social,
#     )
# )


 # 10. Start with a quota-only smoke run



 Before the full stack:



 ```python

 mechanism = quota

 ```



 Use a tiny run:



 ```text

 outer iterations = 2

 candidates        = 2

 seeds             = 1

 horizon           = 10

 agents            = 2

 ```



 This validates the lifecycle before adding additional mechanism channels.

 # 11. Bilevel configuration anatomy



 ```text

 BilevelConfig

 ├── world

 ├── reporting

 ├── mechanism

 ├── outer optimizer: ES

 └── inner optimizer: APPO/RLlib

     ├── environment

     ├── env runners

     ├── learners

     ├── training

     ├── evaluation

     └── agents

 ```

 ## 11.1 Target configuration



 The code below is intentionally conservative and small.



 Align `.mechanism(...)` with the final merged builder signature before

 uncommenting.

In [ ]:
# import ray
# from gymnasium import spaces
#
# from core.callbacks import tag_episode_with_env_idx
# from core.optimizers.bilevel import BilevelConfig
# from core.optimizers.es.config import ESConfig
# from core.optimizers.appo.config import APPOptimizerConfig
# from examples.bilevel_fishery.regulated_env import FisheryRegulatedEnv
# from examples.bilevel_fishery.regulator_env import FisheryRegulatorEnv
#
# ray.shutdown()
#
# bilevel_opt_cfg = (
#     BilevelConfig()
#     .world(world_name="fishery_world")
#     .reporting(
#         reporter="wandb",
#         project_name="bilevel",
#         settings_dict={
#             "x_disable_stats": True,
#             "x_disable_meta": True,
#             "quiet": True,
#             "max_end_of_run_summary_metrics": 0,
#             "max_end_of_run_history_metrics": 0,
#         },
#     )
#     .mechanism(
#         mechanism=mechanism,
#     )
#     .training(
#         outer_iters=2,
#     )
#     .ray(
#         device="cpu",
#         num_cpus=4,
#         omp_threads=1,
#         logging_level="ERROR",
#     )
#     .outer(
#         ESConfig()
#         .training(
#             sigma=0.15,
#             mean_lr=0.10,
#             sigma_decay=1.0,
#             sigma_lr=0.0,
#             min_sigma=0.15,
#             max_sigma=0.15,
#         )
#         .environment(
#             env=FisheryRegulatorEnv,
#             env_config={
#                 "ecology_cfg": {
#                     "sustainability_weight": 2,
#                     "sustainability_threshold": 0.20,
#                     "K": 5_000,
#                 },
#             },
#             horizon=10,
#             train_iters=2,
#         )
#         .debugging(
#             seed=42,
#             num_seeds=1,
#         )
#     )
#     .inner(
#         APPOptimizerConfig()
#         .resources(
#             num_cpus_for_main_process=1,
#         )
#         .framework(
#             framework="torch",
#         )
#         .api_stack(
#             enable_rl_module_and_learner=True,
#             enable_env_runner_and_connector_v2=True,
#         )
#         .environment(
#             env=FisheryRegulatedEnv,
#             env_config={
#                 "ecology_cfg": {
#                     "r": 0.3,
#                     "K": 5_000,
#                     "p": 1.0,
#                     "B0": 4_000,
#                     "fish_init": 4_000,
#                     "sigma": 0.02,
#                     "initial_stock_log_sigma": 0.05,
#                     "unregulated_f_multiplier": 2.0,
#                     "restoration_effectiveness": 0.02,
#                 },
#                 "seed": 0,
#             },
#             horizon=10,
#             disable_env_checking=False,
#         )
#         .env_runners(
#             num_env_runners=0,
#             num_cpus_per_env_runner=1,
#             num_gpus_per_env_runner=0,
#             num_envs_per_env_runner=2,
#             rollout_fragment_length=10,
#             batch_mode="truncate_episodes",
#             max_requests_in_flight_per_env_runner=1,
#         )
#         .learners(
#             num_learners=0,
#             num_gpus_per_learner=0,
#         )
#         .training(
#             vtrace=True,
#             gamma=0.99,
#             lr=0.001,
#             train_batch_size_per_learner=20,
#             minibatch_size=20,
#             num_epochs=1,
#             entropy_coeff=0.001,
#             grad_clip=40.0,
#         )
#         .evaluation(
#             evaluation_interval=1,
#             evaluation_duration=1,
#             evaluation_duration_unit="episodes",
#             evaluation_num_env_runners=0,
#             evaluation_parallel_to_training=False,
#             evaluation_config={
#                 "explore": False,
#                 "rollout_fragment_length": 10,
#                 "batch_mode": "complete_episodes",
#             },
#             base_seed=42,
#             num_seeds=1,
#         )
#         .agents(
#             {
#                 "utilizer": {
#                     "count": 2,
#                     "policy": "fisher_policy",
#                     # Add final action/observation spaces after the mechanism
#                     # stack exposes/validates the transformed dimensions.
#                 }
#             }
#         )
#         .debugging(
#             seed=42,
#             num_seeds=1,
#         )
#         .reporting(
#             min_time_s_per_iteration=0,
#             min_sample_timesteps_per_iteration=0,
#             min_train_timesteps_per_iteration=0,
#         )
#     )
# )
#
# bilevel_opt = bilevel_opt_cfg.build_optimizer()
# bilevel_opt.run()
#
# ray.shutdown()


 # 12. One complete step with the mechanism stack



 Assume:



 ```text

 Quota -> Subsidy -> SocialInfluence

 ```



 and:



 \[

 a_{i,t}=[h_{i,t},e_{i,t}].

 \]



 ## Action phase



 Quota:



 \[

 [h_{i,t},e_{i,t}]

 \mapsto

 [h_{i,t}^{*},e_{i,t}].

 \]



 ## Transition phase



 Harvest:



 \[

 H_t

 =

 \sum_i h_{i,t}^{*}H_{i,t}^{\max}.

 \]



 Restoration:



 \[

 I_t

 =

 \rho\sum_i e_{i,t}.

 \]



 Dynamics:



 \[

 B_{t+1}

 =

 B_t+G(B_t)+\epsilon_t+I_t-H_t.

 \]



 ## Reward phase



 Benchmark computes \(r_{i,t}\).



 Subsidy gives:



 \[

 r_{i,t}^{*}

 =

 r_{i,t}

 -

 ce_{i,t}^2

 +

 \sigma_\theta e_{i,t}.

 \]



 ## Observation phase



 Base observation is built from the new/current benchmark state.



 Quota may append \(\alpha_t\).



 Social influence appends peer previous actions.



 The learner receives the final transformed observation.

 # 13. Reproducibility strategy



 Before comparing training curves:



 1. fix all seeds;

 2. set stochastic growth noise to zero;

 3. feed the same manual action sequence to `dev` and this branch;

 4. compare quota `allowed_frac`;

 5. compare delivered harvest;

 6. compare state transition;

 7. compare reward;

 8. then re-enable stochasticity and RL.



 Action-transform parity is much easier to diagnose than full stochastic RL

 parity.

 # 14. References represented in the project slides



 - Madani & Dinar: exogenous regulatory institutions for common-pool resource

   management.

 - Yang et al. (2022): adaptive incentive design with multi-agent

   meta-gradient reinforcement learning.

 - Jaques et al. (2019): social influence as intrinsic motivation for

   multi-agent deep reinforcement learning.



 The code uses these ideas as mechanism-design patterns. The implemented

 algorithm may be simpler than the full paper method; most importantly, the

 current social mechanism does not yet compute the counterfactual KL reward.